# Classification Metrics - Assignment

Welcome to the assignment for Classification Metrics.

Understanding how to evaluate classification models is fundamental to machine learning. While accuracy provides a simple overview, it can be misleading, especially with imbalanced datasets. Classification metrics like precision, recall, F1-score, and ROC-AUC provide deeper insights into model performance and help you understand where your model succeeds and where it fails.

In this assignment, you will explore various classification metrics, learn when to use each one, and understand the tradeoffs between different evaluation approaches. You'll work with confusion matrices, ROC curves, precision-recall curves, and learn how to optimize classification thresholds for specific business objectives.

Classification metrics are critical in real-world applications where different types of errors have different costs. In medical diagnosis, missing a disease (false negative) may be catastrophic, while in spam detection, marking legitimate emails as spam (false positive) frustrates users. Understanding these metrics allows you to align model performance with business requirements.

**What You Will Do in This Assignment**

* Build and interpret confusion matrices to understand model predictions.
* Calculate precision, recall, and F1-score to evaluate classification quality.
* Create and analyze ROC and Precision-Recall curves.
* Optimize classification thresholds based on business objectives.
* Extend binary metrics to multi-class problems with different averaging strategies.
* Implement classification metrics from scratch using NumPy.

Let's get started!

---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">TIPS FOR SUCCESSFUL GRADING OF YOUR ASSIGNMENT:</h4>

* All cells are frozen except for the ones where you need to submit your solutions or when explicitly mentioned you can interact with it.

* In each exercise cell, look for comments `### START CODE HERE ###` and `### END CODE HERE ###`. These show you where to write the solution code. **Do not add or change any code that is outside these comments**.

* You can add new cells to experiment but these will be omitted by the grader, so don't rely on newly created cells to host your solution code, use the provided places for this.

* Avoid using global variables unless you absolutely have to. The grader tests your code in an isolated environment without running all cells from the top. As a result, global variables may be unavailable when scoring your submission. Global variables that are meant to be used will be defined in UPPERCASE.

* To submit your notebook for grading, first save it by clicking the save icon on the top left of the page and then click on the `Submit assignment` button on the top right of the page.
---

## Table of Contents

1. [Introduction to Classification Metrics](#1)
2. [Confusion Matrix Analysis](#2)
   - [Exercise 1: Build and Analyze Confusion Matrix](#ex-1)
3. [Precision, Recall, and F1 Score](#3)
   - [Exercise 2: Calculate and Interpret Metrics](#ex-2)
4. [ROC and PR Curves](#4)
   - [Exercise 3: Plot and Compare Curves](#ex-3)
5. [Threshold Optimization](#5)
   - [Exercise 4: Optimize for Business Objectives](#ex-4)
6. [Multi-Class Metrics](#6)
   - [Exercise 5: Multi-Class Classification Evaluation](#ex-5)
7. [From-Scratch Implementation](#7)
   - [Exercise 6: Implement Metrics from Scratch](#ex-6)
8. [Conclusion](#8)

<a name='1'></a>
## 1 - Introduction to Classification Metrics

**Classification metrics** evaluate how well a model distinguishes between different classes. Unlike regression metrics that measure prediction error magnitude, classification metrics focus on correctness of class assignments.

### Binary Classification Fundamentals:

For binary classification (positive/negative), predictions fall into four categories:

- **True Positives (TP)**: Correctly predicted positive instances
- **True Negatives (TN)**: Correctly predicted negative instances
- **False Positives (FP)**: Negative instances incorrectly predicted as positive (Type I error)
- **False Negatives (FN)**: Positive instances incorrectly predicted as negative (Type II error)

### Key Metrics:

**Accuracy**: Overall correctness
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision**: Quality of positive predictions
$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall (Sensitivity, TPR)**: Coverage of actual positives
$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-Score**: Harmonic mean of precision and recall
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**Specificity (TNR)**: Coverage of actual negatives
$$\text{Specificity} = \frac{TN}{TN + FP}$$

### Metric Selection:

- **Balanced dataset**: Accuracy works well
- **Imbalanced dataset**: Use precision, recall, F1, or AUC
- **Cost-sensitive**: Optimize based on FP/FN costs
- **Medical diagnosis**: Prioritize recall (avoid missing positives)
- **Spam detection**: Prioritize precision (avoid false alarms)

### Classification Threshold:

Most classifiers output probabilities $P(y=1|x)$. The threshold $\tau$ determines classification:
$$\hat{y} = \begin{cases} 1 & \text{if } P(y=1|x) \geq \tau \\ 0 & \text{otherwise} \end{cases}$$

Default is $\tau = 0.5$, but this can be optimized based on requirements.

### Import Required Libraries

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import unittests
import warnings
warnings.filterwarnings('ignore')

# Machine learning
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    classification_report
)

# Set random seed
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("All libraries imported successfully!")

### Helper Functions

In [ ]:
def plot_confusion_matrix(cm, title="Confusion Matrix", labels=None, normalize=False):
    """Plot confusion matrix with annotations."""
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        fmt = '.2f'
    else:
        fmt = 'd'
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap='Blues', 
                xticklabels=labels, yticklabels=labels,
                cbar_kws={'label': 'Count' if not normalize else 'Proportion'})
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_metrics_vs_threshold(y_true, y_pred_proba, metrics=['precision', 'recall']):
    """Plot how metrics change with classification threshold."""
    thresholds = np.linspace(0, 1, 100)
    metric_values = {metric: [] for metric in metrics}
    
    for threshold in thresholds:
        y_pred = (y_pred_proba >= threshold).astype(int)
        
        if 'precision' in metrics:
            metric_values['precision'].append(
                precision_score(y_true, y_pred, zero_division=0)
            )
        if 'recall' in metrics:
            metric_values['recall'].append(
                recall_score(y_true, y_pred, zero_division=0)
            )
        if 'f1' in metrics:
            metric_values['f1'].append(
                f1_score(y_true, y_pred, zero_division=0)
            )
    
    plt.figure(figsize=(10, 6))
    for metric, values in metric_values.items():
        plt.plot(thresholds, values, linewidth=2, label=metric.capitalize())
    
    plt.xlabel('Threshold', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.title('Metrics vs Classification Threshold', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_roc_curve(y_true, y_pred_proba, label=None):
    """Plot ROC curve with AUC score."""
    fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, linewidth=2, label=f'{label} (AUC = {auc:.3f})' if label else f'AUC = {auc:.3f}')
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('ROC Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return auc


def plot_pr_curve(y_true, y_pred_proba, label=None):
    """Plot Precision-Recall curve."""
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    ap = average_precision_score(y_true, y_pred_proba)
    
    plt.figure(figsize=(10, 8))
    plt.plot(recall, precision, linewidth=2, 
             label=f'{label} (AP = {ap:.3f})' if label else f'AP = {ap:.3f}')
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return ap


print("Helper functions defined!")

### Load Dataset

In [ ]:
# Load breast cancer dataset for binary classification
cancer_data = load_breast_cancer()
X = cancer_data.data
y = cancer_data.target  # 0 = malignant, 1 = benign

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Dataset loaded successfully!")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution (training):")
print(f"  Malignant (0): {(y_train == 0).sum()}")
print(f"  Benign (1): {(y_train == 1).sum()}")

In [ ]:
# Train baseline model
baseline_model = LogisticRegression(max_iter=10000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

# Get predictions
y_pred = baseline_model.predict(X_test_scaled)
y_pred_proba = baseline_model.predict_proba(X_test_scaled)[:, 1]

print("Baseline model trained!")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

<a name='2'></a>
## 2 - Confusion Matrix Analysis

The **confusion matrix** is a fundamental tool for understanding classification performance. It shows the counts of true positives, true negatives, false positives, and false negatives.

### Structure:

```
                Predicted Negative    Predicted Positive
Actual Negative        TN                    FP
Actual Positive        FN                    TP
```

### Derived Metrics:

From the confusion matrix, we can compute:

- **True Positive Rate (TPR/Recall/Sensitivity)**: $\frac{TP}{TP + FN}$
- **True Negative Rate (TNR/Specificity)**: $\frac{TN}{TN + FP}$
- **False Positive Rate (FPR)**: $\frac{FP}{FP + TN} = 1 - \text{Specificity}$
- **False Negative Rate (FNR)**: $\frac{FN}{FN + TP} = 1 - \text{Recall}$

### Interpretation:

- **Diagonal elements** (TN, TP): Correct predictions
- **Off-diagonal elements** (FP, FN): Errors
- **Row-normalized**: Shows proportion of each actual class
- **Column-normalized**: Shows precision for each predicted class

<a name='ex-1'></a>
### Exercise 1 - Build and Analyze Confusion Matrix

**Your Task:**

Implement the `analyze_confusion_matrix` function that:
1. Creates a confusion matrix from predictions
2. Extracts TP, TN, FP, FN values
3. Calculates TPR, TNR, FPR, FNR
4. Returns all metrics in a dictionary

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For confusion matrix:**
* Use: `cm = confusion_matrix(y_true, y_pred)`
* Extract: `tn, fp, fn, tp = cm.ravel()`
* Note: ravel() flattens 2x2 matrix to [TN, FP, FN, TP]

**For rates:**
* TPR = TP / (TP + FN)
* TNR = TN / (TN + FP)
* FPR = FP / (FP + TN) = 1 - TNR
* FNR = FN / (FN + TP) = 1 - TPR

**Edge cases:**
* Handle division by zero with conditional checks
* Return 0.0 or 1.0 as appropriate

</details>

In [ ]:
# GRADED FUNCTION: analyze_confusion_matrix
def analyze_confusion_matrix(y_true, y_pred):
    """
    Analyze confusion matrix and compute all derived metrics.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        Dictionary with cm, tn, fp, fn, tp, tpr, tnr, fpr, fnr
    """
    ### START CODE HERE ### (≈ 15 lines)
    
    # Create confusion matrix
    cm = None  
    
    # Extract TP, TN, FP, FN
    tn, fp, fn, tp = None, None, None, None 
    
    # Calculate rates (handle division by zero)
    tpr = None 
    tnr = None 
    fpr = None 
    fnr = None 
    
    ### END CODE HERE ###
    
    return {
        'confusion_matrix': cm,
        'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
        'tpr': tpr, 'tnr': tnr, 'fpr': fpr, 'fnr': fnr
    }

In [ ]:
# Test your implementation
results = analyze_confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(results['confusion_matrix'])
print(f"\nComponents:")
print(f"  True Negatives (TN):  {results['tn']}")
print(f"  False Positives (FP): {results['fp']}")
print(f"  False Negatives (FN): {results['fn']}")
print(f"  True Positives (TP):  {results['tp']}")
print(f"\nRates:")
print(f"  True Positive Rate (Sensitivity):  {results['tpr']:.4f}")
print(f"  True Negative Rate (Specificity):  {results['tnr']:.4f}")
print(f"  False Positive Rate:               {results['fpr']:.4f}")
print(f"  False Negative Rate:               {results['fnr']:.4f}")

# Visualize
plot_confusion_matrix(results['confusion_matrix'], 
                     title="Confusion Matrix - Breast Cancer Classification",
                     labels=['Malignant', 'Benign'])

In [ ]:
unittests.exercise_1(analyze_confusion_matrix)

##### **Expected Output**
```
Confusion Matrix:
[[XX XX]
 [XX XX]]

Components:
  True Negatives (TN):  XX
  False Positives (FP): XX
  False Negatives (FN): XX
  True Positives (TP):  XX

Rates:
  True Positive Rate (Sensitivity):  0.9XXX
  True Negative Rate (Specificity):  0.9XXX
  False Positive Rate:               0.0XXX
  False Negative Rate:               0.0XXX
```
Your confusion matrix should show high TP and TN values with low FP and FN. The rates should sum appropriately (TPR + FNR = 1, TNR + FPR = 1).

<a name='3'></a>
## 3 - Precision, Recall, and F1 Score

**Precision** and **Recall** provide complementary views of classification performance:

### Precision:
$$\text{Precision} = \frac{TP}{TP + FP}$$

- Answers: "Of all positive predictions, how many were correct?"
- High precision → Few false alarms
- Important when false positives are costly

### Recall:
$$\text{Recall} = \frac{TP}{TP + FN}$$

- Answers: "Of all actual positives, how many did we find?"
- High recall → Few missed cases
- Important when false negatives are costly

### Precision-Recall Tradeoff:

- **Lower threshold** → More positive predictions → Higher recall, lower precision
- **Higher threshold** → Fewer positive predictions → Lower recall, higher precision
- Cannot maximize both simultaneously (usually)

### F1-Score:
$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

- Harmonic mean of precision and recall
- Balances both metrics
- Penalizes extreme imbalance between precision and recall
- Useful when you need a single metric

### F-Beta Score:
$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

- $\beta < 1$: Emphasizes precision
- $\beta = 1$: Balanced (F1-score)
- $\beta > 1$: Emphasizes recall

<a name='ex-2'></a>
### Exercise 2 - Calculate and Interpret Metrics

**Your Task:**

Implement the `calculate_classification_metrics` function that:
1. Calculates precision, recall, and F1 manually from TP/TN/FP/FN
2. Verifies results against sklearn functions
3. Tests different thresholds and observes tradeoffs
4. Recommends optimal threshold based on requirements

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For manual calculation:**
* Precision = TP / (TP + FP)
* Recall = TP / (TP + FN)
* F1 = 2 * (Precision * Recall) / (Precision + Recall)
* Handle division by zero

**For sklearn verification:**
* Use: `precision_score(y_true, y_pred)`
* Use: `recall_score(y_true, y_pred)`
* Use: `f1_score(y_true, y_pred)`

**For threshold testing:**
* Loop through thresholds: `for t in np.linspace(0.1, 0.9, 9):`
* Apply threshold: `y_pred_t = (y_pred_proba >= t).astype(int)`
* Calculate metrics for each

</details>

In [ ]:
# GRADED FUNCTION: calculate_classification_metrics
def calculate_classification_metrics(y_true, y_pred, y_pred_proba=None):
    """
    Calculate precision, recall, and F1 score.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
        y_pred_proba: Predicted probabilities (optional, for threshold analysis)
    
    Returns:
        Dictionary with metrics
    """
    ### START CODE HERE ### (≈ 20 lines)
    
    # Get confusion matrix components
    cm = None  
    tn, fp, fn, tp = None, None, None, None  # Extract from cm
    
    # Calculate metrics manually
    precision_manual = None  
    recall_manual = None
    f1_manual = None
    
    # Verify with sklearn
    precision_sklearn = None  
    recall_sklearn = None  
    f1_sklearn = None 
    
    # Threshold analysis (if probabilities provided)
    threshold_results = None
    if y_pred_proba is not None:
        threshold_results = []
        for threshold in np.linspace(0.1, 0.9, 9):
            # Your code here: apply threshold and calculate metrics
            pass
    
    ### END CODE HERE ###
    
    return {
        'precision_manual': precision_manual,
        'recall_manual': recall_manual,
        'f1_manual': f1_manual,
        'precision_sklearn': precision_sklearn,
        'recall_sklearn': recall_sklearn,
        'f1_sklearn': f1_sklearn,
        'threshold_results': threshold_results
    }

In [ ]:
# Test your implementation
metrics = calculate_classification_metrics(y_test, y_pred, y_pred_proba)

print("Manual Calculations:")
print(f"  Precision: {metrics['precision_manual']:.4f}")
print(f"  Recall:    {metrics['recall_manual']:.4f}")
print(f"  F1 Score:  {metrics['f1_manual']:.4f}")

print("\nSklearn Verification:")
print(f"  Precision: {metrics['precision_sklearn']:.4f}")
print(f"  Recall:    {metrics['recall_sklearn']:.4f}")
print(f"  F1 Score:  {metrics['f1_sklearn']:.4f}")

print("\nDifferences (should be ~0):")
print(f"  Precision: {abs(metrics['precision_manual'] - metrics['precision_sklearn']):.6f}")
print(f"  Recall:    {abs(metrics['recall_manual'] - metrics['recall_sklearn']):.6f}")
print(f"  F1:        {abs(metrics['f1_manual'] - metrics['f1_sklearn']):.6f}")

# Visualize threshold tradeoff
if metrics['threshold_results']:
    df = pd.DataFrame(metrics['threshold_results'])
    print("\nThreshold Analysis:")
    print(df)

# Plot metrics vs threshold
plot_metrics_vs_threshold(y_test, y_pred_proba, metrics=['precision', 'recall', 'f1'])

##### **Expected Output**
```
Manual Calculations:
  Precision: 0.9XXX
  Recall:    0.9XXX
  F1 Score:  0.9XXX

Sklearn Verification:
  Precision: 0.9XXX
  Recall:    0.9XXX
  F1 Score:  0.9XXX

Differences (should be ~0):
  Precision: 0.000000
  Recall:    0.000000
  F1:        0.000000
```
Your manual calculations should match sklearn exactly. The threshold analysis should show precision increasing and recall decreasing as threshold increases.

In [ ]:
unittests.exercise_2(calculate_classification_metrics)

<a name='4'></a>
## 4 - ROC and PR Curves

**ROC (Receiver Operating Characteristic)** and **PR (Precision-Recall)** curves visualize model performance across all classification thresholds.

### ROC Curve:

Plots TPR (Recall) vs FPR at various thresholds:
- **TPR**: $\frac{TP}{TP + FN}$ (y-axis)
- **FPR**: $\frac{FP}{FP + TN}$ (x-axis)

**Interpretation:**
- Diagonal line: Random classifier (AUC = 0.5)
- Top-left corner: Perfect classifier (AUC = 1.0)
- Area Under Curve (AUC): Single-number summary
- AUC = Probability that model ranks random positive higher than random negative

### PR Curve:

Plots Precision vs Recall at various thresholds:
- **Recall**: $\frac{TP}{TP + FN}$ (x-axis)
- **Precision**: $\frac{TP}{TP + FP}$ (y-axis)

**Interpretation:**
- Top-right corner: Perfect classifier
- Baseline: Proportion of positive class
- Average Precision (AP): Area under PR curve
- More informative than ROC for imbalanced datasets

### When to Use Each:

**ROC Curve:**
- Balanced datasets
- Care about both classes equally
- Compare multiple models

**PR Curve:**
- Imbalanced datasets
- Positive class is rare and important
- Focus on positive class performance

<a name='ex-3'></a>
### Exercise 3 - Plot and Compare Curves

**Your Task:**

Implement the `compare_roc_pr_curves` function that:
1. Plots ROC curve with AUC for baseline model
2. Plots PR curve with AP for baseline model
3. Trains a Random Forest model for comparison
4. Plots both models on same ROC and PR curves
5. Analyzes which model performs better

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For ROC curve:**
* Calculate: `fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)`
* AUC: `auc = roc_auc_score(y_true, y_pred_proba)`
* Plot TPR vs FPR
* Add diagonal baseline

**For PR curve:**
* Calculate: `precision, recall, thresholds = precision_recall_curve(y_true, y_pred_proba)`
* AP: `ap = average_precision_score(y_true, y_pred_proba)`
* Plot Precision vs Recall

**For comparison:**
* Train second model: `rf = RandomForestClassifier().fit(X_train, y_train)`
* Get probabilities: `y_pred_proba_rf = rf.predict_proba(X_test)[:, 1]`
* Plot both on same axes with different colors/labels

</details>

In [ ]:
# GRADED FUNCTION: compare_roc_pr_curves
def compare_roc_pr_curves(X_train, X_test, y_train, y_test, model1, model2=None):
    """
    Compare ROC and PR curves for one or two models.
    
    Args:
        X_train: Training features
        X_test: Test features
        y_train: Training labels
        y_test: Test labels
        model1: First trained model
        model2: Second model class (will be trained)
    
    Returns:
        Dictionary with metrics for both models
    """
    ### START CODE HERE ### (≈ 35 lines)
    
    # Get predictions from model1
    y_pred_proba1 = None 
    
    # Calculate ROC for model1
    fpr1, tpr1, _ = None, None, None  
    auc1 = None  
    
    # Calculate PR for model1
    precision1, recall1, _ = None, None, None  
    ap1 = None  
    
    results = {
        'model1': {
            'fpr': fpr1, 'tpr': tpr1, 'auc': auc1,
            'precision': precision1, 'recall': recall1, 'ap': ap1
        }
    }
    
    # Train and evaluate model2 if provided
    if model2 is not None:
        # Your code here: train model2, get predictions, calculate metrics
        pass
    
    # Plot ROC curves
    plt.figure(figsize=(12, 5))
    
    # ROC subplot
    plt.subplot(1, 2, 1)
    # Your plotting code here
    
    # PR subplot
    plt.subplot(1, 2, 2)
    # Your plotting code here
    
    plt.tight_layout()
    plt.show()
    
    ### END CODE HERE ###
    
    return results

In [ ]:
# Compare baseline vs Random Forest
print("="*60)
print("ROC AND PR CURVE COMPARISON")
print("="*60)

comparison_results = compare_roc_pr_curves(
    X_train_scaled, X_test_scaled, y_train, y_test,
    model1=baseline_model,
    model2=RandomForestClassifier(n_estimators=100, random_state=42)
)

print("\nMetrics Comparison:")
print(f"{'Model':<20} {'AUC-ROC':<12} {'AP-PR':<12}")
print("-"*44)
for name, metrics in comparison_results.items():
    print(f"{name:<20} {metrics['auc']:<12.4f} {metrics['ap']:<12.4f}")

##### **Expected Output**
```
============================================================
ROC AND PR CURVE COMPARISON
============================================================

Metrics Comparison:
Model                AUC-ROC      AP-PR       
--------------------------------------------
model1               0.9XXX       0.9XXX      
model2               0.9XXX       0.9XXX      
```
You should see two plots side by side: ROC curves (left) and PR curves (right). Both models should perform well on this dataset, with curves well above baseline.

In [ ]:
unittests.exercise_3(compare_roc_pr_curves)

<a name='5'></a>
## 5 - Threshold Optimization

The default classification threshold of 0.5 is often suboptimal. The optimal threshold depends on:

### Business Objectives:

**Medical Diagnosis:**
- False Negative (missing disease): Very costly
- False Positive (unnecessary tests): Less costly
- **Solution**: Lower threshold, prioritize recall

**Spam Detection:**
- False Positive (blocking important email): Very costly
- False Negative (spam in inbox): Less costly
- **Solution**: Higher threshold, prioritize precision

### Cost-Sensitive Classification:

Define cost function:
$$\text{Cost} = C_{FP} \times FP + C_{FN} \times FN$$

Find threshold that minimizes total cost:
$$\tau^* = \arg\min_{\tau} \text{Cost}(\tau)$$

### Optimization Methods:

1. **Cost Minimization**: Minimize weighted sum of errors
2. **F-Beta Maximization**: Maximize F-score with custom $\beta$
3. **Youden's Index**: Maximize $J = \text{TPR} - \text{FPR}$
4. **Target Metric**: Achieve specific recall/precision level

### Practical Considerations:

- Validate threshold on separate dataset
- Monitor performance over time
- Consider uncertainty in cost estimates
- Document decision rationale

<a name='ex-4'></a>
### Exercise 4 - Optimize for Business Objectives

**Your Task:**

Implement the `optimize_threshold` function that:
1. Defines cost scenarios (medical, balanced, spam)
2. Tests thresholds from 0.01 to 0.99
3. Calculates total cost for each threshold
4. Finds optimal threshold for each scenario
5. Compares metrics at optimal thresholds

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For cost calculation:**
* Apply threshold: `y_pred_t = (y_pred_proba >= threshold).astype(int)`
* Get confusion matrix: `tn, fp, fn, tp = confusion_matrix(y_true, y_pred_t).ravel()`
* Calculate cost: `cost = cost_fp * fp + cost_fn * fn`

**For optimization:**
* Test range: `for threshold in np.linspace(0.01, 0.99, 99):`
* Find minimum: `optimal_idx = np.argmin(costs)`
* Optimal threshold: `thresholds[optimal_idx]`

**For scenarios:**
* Medical: `cost_fn = 100, cost_fp = 1`
* Balanced: `cost_fn = 10, cost_fp = 10`
* Spam: `cost_fn = 1, cost_fp = 100`

</details>

In [ ]:
# GRADED FUNCTION: optimize_threshold
def optimize_threshold(y_true, y_pred_proba, cost_fp=1, cost_fn=1):
    """
    Find optimal classification threshold based on cost function.
    
    Args:
        y_true: True labels
        y_pred_proba: Predicted probabilities
        cost_fp: Cost of false positive
        cost_fn: Cost of false negative
    
    Returns:
        Dictionary with optimal threshold and metrics
    """
    ### START CODE HERE ### (≈ 25 lines)
    
    thresholds = np.linspace(0.01, 0.99, 99)
    costs = []
    metrics_list = []
    
    for threshold in thresholds:
        # Apply threshold
        y_pred = None  
        
        # Get confusion matrix components
        cm = None
        tn, fp, fn, tp = None, None, None, None
        
        # Calculate cost
        cost = None  
        costs.append(cost)
        
        # Calculate metrics
        precision = None
        recall = None
        f1 = None
        
        metrics_list.append({
            'threshold': threshold,
            'cost': cost,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'fp': fp,
            'fn': fn
        })
    
    # Find optimal threshold
    optimal_idx = None  # Index of minimum cost
    optimal_threshold = None
    optimal_metrics = None
    
    ### END CODE HERE ###
    
    return {
        'optimal_threshold': optimal_threshold,
        'optimal_metrics': optimal_metrics,
        'all_metrics': metrics_list,
        'thresholds': thresholds,
        'costs': costs
    }

In [ ]:
# Test different cost scenarios
print("="*60)
print("THRESHOLD OPTIMIZATION")
print("="*60)

scenarios = {
    'Medical (High Recall)': {'cost_fp': 1, 'cost_fn': 100},
    'Balanced': {'cost_fp': 10, 'cost_fn': 10},
    'Spam (High Precision)': {'cost_fp': 100, 'cost_fn': 1}
}

results_comparison = []

for scenario_name, costs in scenarios.items():
    print(f"\n{scenario_name}:")
    print(f"  Cost FP={costs['cost_fp']}, Cost FN={costs['cost_fn']}")
    
    result = optimize_threshold(y_test, y_pred_proba, **costs)
    
    opt = result['optimal_metrics']
    print(f"  Optimal Threshold: {result['optimal_threshold']:.3f}")
    print(f"  Precision: {opt['precision']:.4f}")
    print(f"  Recall:    {opt['recall']:.4f}")
    print(f"  F1 Score:  {opt['f1']:.4f}")
    print(f"  Total Cost: {opt['cost']:.2f}")
    
    results_comparison.append({
        'Scenario': scenario_name,
        'Threshold': result['optimal_threshold'],
        'Precision': opt['precision'],
        'Recall': opt['recall'],
        'F1': opt['f1']
    })

# Compare scenarios
print("\n" + "="*60)
print("SCENARIO COMPARISON")
print("="*60)
comparison_df = pd.DataFrame(results_comparison)
print(comparison_df.to_string(index=False))

# Visualize cost vs threshold
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for scenario_name, costs in scenarios.items():
    result = optimize_threshold(y_test, y_pred_proba, **costs)
    axes[0].plot(result['thresholds'], result['costs'], 
                linewidth=2, label=scenario_name, alpha=0.7)
    axes[0].scatter([result['optimal_threshold']], [result['optimal_metrics']['cost']], 
                   s=100, marker='*', zorder=5)

axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Total Cost', fontsize=12)
axes[0].set_title('Cost vs Threshold', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Metrics comparison
x = np.arange(len(results_comparison))
width = 0.25
axes[1].bar(x - width, comparison_df['Precision'], width, label='Precision', alpha=0.8)
axes[1].bar(x, comparison_df['Recall'], width, label='Recall', alpha=0.8)
axes[1].bar(x + width, comparison_df['F1'], width, label='F1', alpha=0.8)
axes[1].set_xlabel('Scenario', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Metrics at Optimal Thresholds', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(['Medical', 'Balanced', 'Spam'], rotation=45)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

##### **Expected Output**
```
============================================================
THRESHOLD OPTIMIZATION
============================================================

Medical (High Recall):
  Cost FP=1, Cost FN=100
  Optimal Threshold: 0.XXX
  Precision: 0.9XXX
  Recall:    0.9XXX  (should be high)
  F1 Score:  0.9XXX
  Total Cost: XX.XX

Balanced:
  Cost FP=10, Cost FN=10
  Optimal Threshold: 0.XXX (should be ~0.5)
  ...

Spam (High Precision):
  Cost FP=100, Cost FN=1
  Optimal Threshold: 0.XXX
  Precision: 0.9XXX  (should be high)
  Recall:    0.9XXX
  ...
```
Medical scenario should have lower threshold (high recall), spam scenario should have higher threshold (high precision).

In [ ]:
unittests.exercise_4(optimize_threshold)

<a name='6'></a>
## 6 - Multi-Class Metrics

Extending binary classification metrics to multiple classes requires aggregation strategies.

### Averaging Methods:

**Macro Average:**
$$\text{Macro-Precision} = \frac{1}{K} \sum_{k=1}^{K} \text{Precision}_k$$

- Calculate metric for each class independently
- Take unweighted average
- Treats all classes equally
- Good for balanced datasets
- Sensitive to minority class performance

**Micro Average:**
$$\text{Micro-Precision} = \frac{\sum_{k=1}^{K} TP_k}{\sum_{k=1}^{K} (TP_k + FP_k)}$$

- Aggregate contributions of all classes
- Calculate metric from aggregated TP, FP, FN
- Each sample contributes equally
- Good for imbalanced datasets
- Micro-F1 equals accuracy in multi-class

**Weighted Average:**
$$\text{Weighted-Precision} = \sum_{k=1}^{K} w_k \cdot \text{Precision}_k$$

where $w_k = \frac{n_k}{n}$ (proportion of class k)

- Weight each class by its frequency
- Accounts for class imbalance
- Most commonly used in practice

### Multi-Class Confusion Matrix:

- $K \times K$ matrix for K classes
- Diagonal: Correct predictions
- Off-diagonal: Confusions between specific class pairs
- Useful for identifying which classes are confused

### One-vs-Rest (OvR):

For each class k:
- Positive: Class k
- Negative: All other classes
- Calculate binary metrics
- Aggregate using macro/micro/weighted

<a name='ex-5'></a>
### Exercise 5 - Multi-Class Classification Evaluation

**Your Task:**

Implement the `evaluate_multiclass` function that:
1. Loads Iris dataset (3 classes)
2. Trains a multi-class classifier
3. Creates confusion matrix
4. Calculates metrics with macro/micro/weighted averaging
5. Generates detailed classification report

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For loading data:**
* Use: `iris = load_iris()`
* Features: `iris.data`, Labels: `iris.target`
* Class names: `iris.target_names`

**For metrics with averaging:**
* Precision: `precision_score(y_true, y_pred, average='macro')`
* Same for recall, f1
* Try: 'macro', 'micro', 'weighted'
* Without average: returns array of per-class metrics

**For classification report:**
* Use: `classification_report(y_true, y_pred, target_names=class_names)`
* Returns formatted string with all metrics

</details>

In [ ]:
# GRADED FUNCTION: evaluate_multiclass
def evaluate_multiclass(model_class=LogisticRegression):
    """
    Evaluate multi-class classification with different averaging methods.
    
    Args:
        model_class: Classifier class to use
    
    Returns:
        Dictionary with confusion matrix and metrics
    """
    ### START CODE HERE ### (≈ 30 lines)
    
    # Load Iris dataset
    iris = None  # load_iris()
    X, y = None, None
    class_names = None  # iris.target_names
    
    # Split data
    X_train, X_test, y_train, y_test = None, None, None, None  # train_test_split
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = None  
    X_test_scaled = None  
    
    # Train model
    model = None  # Instantiate and fit
    
    # Get predictions
    y_pred = None
    
    # Confusion matrix
    cm = None
    
    # Calculate metrics with different averaging
    metrics = {}
    for avg_method in ['macro', 'micro', 'weighted', None]:
        # Your code here: calculate precision, recall, f1
        pass
    
    # Classification report
    report = None  
    
    ### END CODE HERE ###
    
    return {
        'confusion_matrix': cm,
        'class_names': class_names,
        'metrics': metrics,
        'classification_report': report,
        'y_test': y_test,
        'y_pred': y_pred
    }

In [ ]:
# Evaluate multi-class classification
print("="*60)
print("MULTI-CLASS CLASSIFICATION EVALUATION")
print("="*60)

mc_results = evaluate_multiclass(LogisticRegression)

# Confusion matrix
print("\nConfusion Matrix:")
print(mc_results['confusion_matrix'])

# Visualize confusion matrix
plot_confusion_matrix(mc_results['confusion_matrix'],
                     title="Multi-Class Confusion Matrix - Iris Dataset",
                     labels=mc_results['class_names'])

# Normalized confusion matrix
plot_confusion_matrix(mc_results['confusion_matrix'],
                     title="Normalized Confusion Matrix - Iris Dataset",
                     labels=mc_results['class_names'],
                     normalize=True)

# Metrics comparison
print("\nMetrics Comparison:")
print(f"{'Averaging':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-"*51)
for avg_method, metric_dict in mc_results['metrics'].items():
    avg_name = str(avg_method) if avg_method else 'per-class'
    if avg_method is None:
        # Per-class metrics
        for i, class_name in enumerate(mc_results['class_names']):
            print(f"  {class_name:<13} {metric_dict['precision'][i]:<12.4f} "
                  f"{metric_dict['recall'][i]:<12.4f} {metric_dict['f1'][i]:<12.4f}")
    else:
        print(f"{avg_name:<15} {metric_dict['precision']:<12.4f} "
              f"{metric_dict['recall']:<12.4f} {metric_dict['f1']:<12.4f}")

# Classification report
print("\n" + "="*60)
print("DETAILED CLASSIFICATION REPORT")
print("="*60)
print(mc_results['classification_report'])

##### **Expected Output**
```
============================================================
MULTI-CLASS CLASSIFICATION EVALUATION
============================================================

Confusion Matrix:
[[XX  X  X]
 [ X XX  X]
 [ X  X XX]]

Metrics Comparison:
Averaging       Precision    Recall       F1-Score    
---------------------------------------------------
macro           0.9XXX       0.9XXX       0.9XXX      
micro           0.9XXX       0.9XXX       0.9XXX      
weighted        0.9XXX       0.9XXX       0.9XXX      
  setosa        0.9XXX       0.9XXX       0.9XXX      
  versicolor    0.9XXX       0.9XXX       0.9XXX      
  virginica     0.9XXX       0.9XXX       0.9XXX      
```
Your confusion matrix should be mostly diagonal. Micro-F1 should equal accuracy. Per-class metrics show performance for each individual class.

In [ ]:
unittests.exercise_5(evaluate_multiclass)

<a name='7'></a>
## 7 - From-Scratch Implementation: Confusion Matrix and Metrics

Implementing classification metrics from scratch deepens understanding of their mathematical foundations.

### Core Implementation:

All metrics derive from the confusion matrix:

```python
# Binary classification
TP = ((y_true == 1) & (y_pred == 1)).sum()
TN = ((y_true == 0) & (y_pred == 0)).sum()
FP = ((y_true == 0) & (y_pred == 1)).sum()
FN = ((y_true == 1) & (y_pred == 0)).sum()
```

### Key Considerations:

1. **Division by Zero**: Handle cases where denominators are zero
2. **Data Types**: Ensure proper numpy array handling
3. **Edge Cases**: Empty predictions, all-positive/all-negative
4. **Numerical Stability**: Use appropriate precision

### Validation:

- Test against sklearn implementations
- Use simple known cases first
- Check edge cases
- Verify mathematical relationships

<a name='ex-6'></a>
### Exercise 6 - Implement Metrics from Scratch

**Your Task:**

Implement from scratch:
1. `confusion_matrix_scratch(y_true, y_pred)`
2. `accuracy_scratch(y_true, y_pred)`
3. `precision_scratch(y_true, y_pred)`
4. `recall_scratch(y_true, y_pred)`
5. `f1_scratch(y_true, y_pred)`
6. Validate against sklearn

<details>
  <summary><b><font color="green">Code Hints (Click to expand if you need help)</font></b></summary>
  
**For confusion matrix:**
* Use boolean indexing: `(y_true == 0) & (y_pred == 0)`
* Count with `.sum()`
* Return 2x2 numpy array: `np.array([[tn, fp], [fn, tp]])`

**For metrics:**
* Extract TP, TN, FP, FN from confusion matrix
* Apply formulas:
  - Accuracy: (TP + TN) / (TP + TN + FP + FN)
  - Precision: TP / (TP + FP)
  - Recall: TP / (TP + FN)
  - F1: 2 * (Precision * Recall) / (Precision + Recall)

**For edge cases:**
* Check if denominator is zero before division
* Return 0.0 or 1.0 as appropriate
* Use: `if (tp + fp) == 0: return 0.0`

</details>

#### 6a. `confusion_matrix_scratch` - Confusion Matrix

The confusion matrix is the foundation of all binary classification metrics. It counts the four possible outcomes when comparing true labels against predicted labels.

$$CM = \begin{bmatrix} TN & FP \\ FN & TP \end{bmatrix}$$

where **TP** = true positives, **TN** = true negatives, **FP** = false positives (Type I error), and **FN** = false negatives (Type II error). Return a $2 \times 2$ numpy array.

In [ ]:
# GRADED FUNCTION: confusion_matrix_scratch
def confusion_matrix_scratch(y_true, y_pred):
    """
    Create confusion matrix from scratch.
    
    Args:
        y_true: True labels (numpy array)
        y_pred: Predicted labels (numpy array)
    
    Returns:
        2x2 confusion matrix [[TN, FP], [FN, TP]]
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    # Convert to numpy arrays
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate TP, TN, FP, FN using boolean indexing
    tp = None  
    tn = None  
    fp = None  
    fn = None  
    
    # Create confusion matrix
    cm = None 
    
    ### END CODE HERE ###
    
    return cm

In [ ]:
unittests.exercise_6a(confusion_matrix_scratch)

#### 6b. `accuracy_scratch` - Accuracy

Accuracy measures the overall fraction of correct predictions. While intuitive, it can be misleading on imbalanced datasets — a model that always predicts the majority class can achieve high accuracy while being useless.

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

In [ ]:
# GRADED FUNCTION: accuracy_scratch
def accuracy_scratch(y_true, y_pred):
    """
    Calculate accuracy from scratch.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        Accuracy score (float)
    """
    ### START CODE HERE ### (≈ 5 lines)
    
    cm = None # confusion_matrix
    tn, fp, fn, tp = None, None, None, None  # Extract from cm
    
    accuracy = None
    
    ### END CODE HERE ###
    
    return accuracy

In [ ]:
unittests.exercise_6b(accuracy_scratch)

#### 6c. `precision_scratch` - Precision

Precision measures the quality of positive predictions: of all samples the model labelled positive, what fraction actually were? It is the metric to optimise when **false positives are costly** (e.g. spam filters).

$$\text{Precision} = \frac{TP}{TP + FP}$$

In [ ]:
# GRADED FUNCTION: precision_scratch
def precision_scratch(y_true, y_pred):
    """
    Calculate precision from scratch.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        Precision score (float)
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    cm = None # confusion_matrix
    tn, fp, fn, tp = None, None, None, None  # Extract from cm
    
    # Handle division by zero
    if (tp + fp) == 0:
        return 0.0
    
    precision = None  
    
    ### END CODE HERE ###
    
    return precision

In [ ]:
unittests.exercise_6c(precision_scratch)

#### 6d. `recall_scratch` - Recall

Recall (also called Sensitivity or True Positive Rate) measures the coverage of actual positives: of all truly positive samples, what fraction did the model catch? It is the metric to optimise when **false negatives are costly** (e.g. medical diagnosis).

$$\text{Recall} = \frac{TP}{TP + FN}$$

In [ ]:
# GRADED FUNCTION: recall_scratch
def recall_scratch(y_true, y_pred):
    """
    Calculate recall from scratch.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        Recall score (float)
    """
    ### START CODE HERE ### (≈ 8 lines)
    
    cm = None # confusion_matrix
    tn, fp, fn, tp = None, None, None, None  # Extract from cm
    
    # Handle division by zero
    if (tp + fn) == 0:
        return 0.0
    
    recall = None  
    
    ### END CODE HERE ###
    
    return recall

In [ ]:
unittests.exercise_6d(recall_scratch)

#### 6e. `f1_scratch` - F1 Score

The F1 score is the **harmonic mean** of precision and recall. Because it uses the harmonic mean, it is low whenever either metric is low — making it a good single summary when you care about both false positives and false negatives.

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

In [ ]:
# GRADED FUNCTION: f1_scratch
def f1_scratch(y_true, y_pred):
    """
    Calculate F1 score from scratch.
    
    Args:
        y_true: True labels
        y_pred: Predicted labels
    
    Returns:
        F1 score (float)
    """
    ### START CODE HERE ### (≈ 10 lines)
    
    precision = None
    recall = None
    
    # Handle division by zero
    if (precision + recall) == 0:
        return 0.0
    
    f1 = None  
    
    ### END CODE HERE ###
    
    return f1

In [ ]:
unittests.exercise_6e(f1_scratch)

In [ ]:
# Test implementations
print("="*60)
print("FROM-SCRATCH IMPLEMENTATION VALIDATION")
print("="*60)

# Test with simple example first
y_true_test = np.array([0, 0, 1, 1, 1, 0, 1, 0])
y_pred_test = np.array([0, 1, 1, 1, 0, 0, 1, 0])

print("\nSimple Test Case:")
print(f"y_true: {y_true_test}")
print(f"y_pred: {y_pred_test}")

print("\nConfusion Matrix:")
cm_scratch = confusion_matrix_scratch(y_true_test, y_pred_test)
cm_sklearn = confusion_matrix(y_true_test, y_pred_test)
print(f"Scratch:\n{cm_scratch}")
print(f"Sklearn:\n{cm_sklearn}")
print(f"Match: {np.array_equal(cm_scratch, cm_sklearn)}")

# Test on actual dataset
print("\n" + "="*60)
print("BREAST CANCER DATASET VALIDATION")
print("="*60)

metrics_scratch = {
    'Accuracy': accuracy_scratch(y_test, y_pred),
    'Precision': precision_scratch(y_test, y_pred),
    'Recall': recall_scratch(y_test, y_pred),
    'F1': f1_scratch(y_test, y_pred)
}

metrics_sklearn = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred)
}

print(f"\n{'Metric':<12} {'Scratch':<12} {'Sklearn':<12} {'Difference':<12} {'Match'}")
print("-"*60)
for metric_name in metrics_scratch:
    scratch_val = metrics_scratch[metric_name]
    sklearn_val = metrics_sklearn[metric_name]
    diff = abs(scratch_val - sklearn_val)
    match = '✓' if diff < 1e-10 else '✗'
    print(f"{metric_name:<12} {scratch_val:<12.6f} {sklearn_val:<12.6f} {diff:<12.2e} {match}")

# Overall validation
all_match = all(abs(metrics_scratch[m] - metrics_sklearn[m]) < 1e-10 for m in metrics_scratch)
print("\n" + "="*60)
if all_match:
    print("✓ ALL IMPLEMENTATIONS CORRECT!")
else:
    print("✗ Some implementations need fixing")
print("="*60)

##### **Expected Output**
```
============================================================
FROM-SCRATCH IMPLEMENTATION VALIDATION
============================================================

Simple Test Case:
y_true: [0 0 1 1 1 0 1 0]
y_pred: [0 1 1 1 0 0 1 0]

Confusion Matrix:
Scratch:
[[3 1]
 [1 3]]
Sklearn:
[[3 1]
 [1 3]]
Match: True

============================================================
BREAST CANCER DATASET VALIDATION
============================================================

Metric       Scratch      Sklearn      Difference   Match
------------------------------------------------------------
Accuracy     0.9XXXXX     0.9XXXXX     X.XXe-XX     ✓
Precision    0.9XXXXX     0.9XXXXX     X.XXe-XX     ✓
Recall       0.9XXXXX     0.9XXXXX     X.XXe-XX     ✓
F1           0.9XXXXX     0.9XXXXX     X.XXe-XX     ✓

============================================================
✓ ALL IMPLEMENTATIONS CORRECT!
============================================================
```
All your from-scratch implementations should match sklearn exactly (difference < 1e-10).

<a name='8'></a>
## 8 - Conclusion

**Congratulations!** You've completed the Classification Metrics assignment!

### What You've Learned:

1. **Confusion Matrix**: Foundation for understanding classification performance at granular level
2. **Precision and Recall**: Complementary metrics with inherent tradeoff
3. **ROC and PR Curves**: Visualizing performance across all thresholds
4. **Threshold Optimization**: Aligning model decisions with business objectives
5. **Multi-Class Metrics**: Extending binary metrics with proper averaging
6. **From-Scratch Implementation**: Deep understanding through manual calculation

### Key Takeaways:

**Metric Selection:**
- **Accuracy**: Only for balanced datasets
- **Precision**: When false positives are costly (spam, recommendations)
- **Recall**: When false negatives are costly (medical, security)
- **F1**: Balance between precision and recall
- **AUC-ROC**: Overall model discrimination ability
- **AUC-PR**: Better for imbalanced datasets

**Best Practices:**
- Always examine confusion matrix first
- Consider class imbalance
- Optimize threshold for business objectives
- Use multiple metrics for comprehensive evaluation
- Validate on separate test set
- Document metric choices and rationale

**Common Pitfalls:**

1. **Relying solely on accuracy** for imbalanced datasets
   - Solution: Use precision, recall, F1, or AUC

2. **Ignoring classification threshold**
   - Solution: Optimize threshold based on costs

3. **Wrong metric for problem**
   - Solution: Align metric with business requirements

4. **Overfitting to validation metrics**
   - Solution: Use separate test set for final evaluation

5. **Comparing models with different thresholds**
   - Solution: Use probability-based metrics (AUC)

### Real-World Applications:

**Medical Diagnosis:**
- High recall to avoid missing diseases
- Accept lower precision (more false alarms)
- Use PR curves for rare diseases

**Fraud Detection:**
- Balance precision and recall
- Minimize false positives (customer annoyance)
- Minimize false negatives (financial loss)

**Information Retrieval:**
- Precision at K for search engines
- Mean Average Precision for ranking
- NDCG for graded relevance

**Spam Detection:**
- High precision (don't block legitimate email)
- Acceptable recall (some spam gets through)

### Advanced Topics:

**Cost-Sensitive Learning:**
- Incorporate costs directly into training
- MetaCost, SMOTE, class weights

**Calibration:**
- Ensure predicted probabilities are accurate
- Platt scaling, isotonic regression
- Reliability diagrams

**Ranking Metrics:**
- Mean Average Precision (MAP)
- Normalized Discounted Cumulative Gain (NDCG)
- Mean Reciprocal Rank (MRR)

**Multi-Label Classification:**
- Hamming loss, exact match ratio
- Label-based and example-based metrics

### Tools and Libraries:

**Evaluation:**
- scikit-learn: Comprehensive metrics library
- Yellowbrick: Visualization for model evaluation
- MLxtend: Additional metrics and utilities

**Monitoring:**
- Evidently: ML monitoring and drift detection
- WhyLogs: Data quality monitoring
- Great Expectations: Data validation

**Experiment Tracking:**
- MLflow: Track metrics across experiments
- Weights & Biases: Real-time metric visualization
- Neptune.ai: Metadata management

### Recommended Resources:

- **Papers**:
  - "The Precision-Recall Plot Is More Informative" (Davis & Goadrich, 2006)
  - "ROC Graphs" (Fawcett, 2006)
  - "Learning from Imbalanced Data" (He & Garcia, 2009)

- **Books**:
  - "Evaluating Machine Learning Models" (Zheng, 2015)
  - "Imbalanced Learning" (He & Ma, 2013)

- **Documentation**:
  - scikit-learn metrics guide
  - Kaggle evaluation metrics guide

**Excellent work! You now understand how to properly evaluate classification models across diverse scenarios and requirements!**